In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [2]:
groups = [
    "NCD", "BL", "NAD", "AI", "NAC", "ND",
    "CS", "AT", "NA", "ADL", "AS_NA", "AS_NAC"
]

columns = [f"{group}_{t}" for group in groups for t in range(8)]
columns.append("target")

df = pd.read_csv(
    "TomsHardware.data",
    header=None,
    names=columns
)

df["target_log"] = np.log1p(df["target"])
print(df.shape)
#df.head(20)

(28179, 98)


In [3]:
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. target log
df["target_log"] = np.log1p(df["target"])

# 2. X і y (без leakage ND)
X = df.drop(
    ["target", "target_log"] + [col for col in df.columns if col.startswith("ND_")],
    axis=1
)
y = df["target_log"]

# 3. split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. базова модель
svr = SVR()

# 6. невелика сітка параметрів
param_grid = {
    "kernel": ["rbf"],
    "C": [1, 10, 50],
    "epsilon": [0.1, 0.2, 0.5],
    "gamma": ["scale", "auto"]
}

# 7. grid search
grid = GridSearchCV(
    estimator=svr,
    param_grid=param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_scaled, y_train)

# 8. найкраща модель
best_model = grid.best_estimator_
print("Best params:", grid.best_params_)

# 9. predict
y_pred = best_model.predict(X_test_scaled)

# 10. метрики в log-scale
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\nMetrics in log-scale:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")

Fitting 3 folds for each of 18 candidates, totalling 54 fits
Best params: {'C': 10, 'epsilon': 0.2, 'gamma': 'auto', 'kernel': 'rbf'}

Metrics in log-scale:
RMSE: 1.2396
MAE:  0.9009
R²:   0.7272
